# 04 · Write the analysis as a sentence

## Context

A collaborator needs to review a transformation before trusting its result.
They should be able to distinguish the input, each scientific operation, and
the point where ordinary Python resumes without reading framework machinery.

## Question

Can a complete method remain both compact and explicit?

## Analysis story

We will compare direct and piped calls, then compose a built-in anomaly with an
ordinary project function. The pipe is the readable analytical sentence; the
surrounding cells provide evidence and interpretation.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v

# This signal combines a repeating temporal pattern, a spatial offset, and a
# tiny seeded noise term so each operation has something visible to transform.
rng = np.random.default_rng(42)
time = pd.date_range("2024-01-01", periods=12, freq="MS")
season = np.sin(np.linspace(0, 2 * np.pi, time.size, endpoint=False))[:, None, None]
spatial = np.array([[0.0, 0.2, 0.4], [0.1, 0.3, 0.5]])[None, :, :]
cube = xr.DataArray(
    season + spatial + rng.normal(0, 0.02, size=(12, 2, 3)),
    dims=("time", "y", "x"),
    coords={"time": time, "y": [40.1, 40.0], "x": [-105.2, -105.1, -105.0]},
    name="environmental_signal",
    attrs={"units": "1", "source": "deterministic synthetic vignette"},
)
cube

## Pipe · Read from top to bottom

`pipe(cube)` supplies the subject. Each verb adds one operation. `unwrap()`
marks the return to xarray and ordinary Python.

In [ ]:
# Direct and piped calls use the same verb contract.
direct = v.zscore(dim="time")(cube)
through_pipe = (
    pipe(cube)
    | v.zscore(dim="time")
).unwrap()

# v.apply lets a compatible project function join the same sentence.
scaled_anomaly = (
    pipe(cube)
    | v.anomaly(dim="time")
    | v.apply(lambda value, factor: value * factor, factor=1.5)
).unwrap()

# This is a compact regression test embedded in the lesson.
xr.testing.assert_allclose(direct, through_pipe)

## Figure · Verify meaning, not just syntax

Compare the same location before and after standardization, then check that the
composed anomaly still preserves the spatial cube.

In [ ]:
import matplotlib.pyplot as plt

# Hold location constant so the first two panels compare like with like; use a
# map in the third panel to show that the full spatial cube was preserved.
site = {"y": 40.0, "x": -105.1}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
cube.sel(**site).plot(ax=axes[0], color="#8b543c")
axes[0].set_title("Input cube at one pixel")
through_pipe.sel(**site).plot(ax=axes[1], color="#3f6f72")
axes[1].axhline(0, color="0.4", linewidth=0.8)
axes[1].set_title("Direct call = pipe call")
scaled_anomaly.isel(time=3).plot(ax=axes[2], cmap="RdBu_r", center=0)
axes[2].set_title("v.anomaly | v.apply")
plt.show()

## What the figure tells us

The standardized series retains timing while expressing values on a comparable
scale. The map shows that chaining `v.anomaly` and `v.apply` preserved spatial
structure. The code reads like the method because each line adds one idea.

## Try the next variation

Change the `factor` passed to `v.apply`, or insert `v.mean` at the end. Predict
the output shape before running the cell.